In [38]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
import sympy as sp
from sympy import symbols, Matrix, exp, sin, cos, pi, diff, simplify

# 设置符号打印
sp.init_printing(use_latex=True)

# State Space Method 2: Displacement-Base

In [39]:
# 定义所有符号变量
# 坐标变量
x, z, xi, eta = symbols('x z xi eta')
l, h = symbols('l h')  # 梁长和厚度

# 材料参数
lambd = symbols('lambda')  # 梯度指数
c11_0, c13_0, c33_0, c55_0 = symbols('c11^0 c13^0 c33^0 c55^0')

# 位移变量
u, w = symbols('u w')
U_n, W_n = symbols('U_n W_n')
n = symbols('n')  # 级数项数

In [31]:
print("\n2. 位移函数假设")
print("=" * 60)

# 位移函数形式（满足简支边界条件）
u_expr = h * U * sp.cos(n * sp.pi * xi)
w_expr = h * W * sp.sin(n * sp.pi * xi)

print("位移函数假设:")
print(f"u(x,z) = {u_expr}")
print(f"w(x,z) = {w_expr}")
print(f"其中 ξ = x/L, η = z/h")

# 坐标变换关系
coord_subs = {x: xi*L, z: eta*h}


2. 位移函数假设
位移函数假设:
u(x,z) = h*U(eta)*cos(pi*n*xi)
w(x,z) = h*W(eta)*sin(pi*n*xi)
其中 ξ = x/L, η = z/h


In [32]:
print("\n4. 应力-应变关系（本构方程）")
print("=" * 60)

# 注意：材料常数是z的函数，通过η表示
c11_eta = c11(eta*h)
c13_eta = c13(eta*h) 
c33_eta = c33(eta*h)
c55_eta = c55(eta*h)

# 应力表达式
sigma_x = c11_eta * epsilon_x + c13_eta * epsilon_z
sigma_z = c13_eta * epsilon_x + c33_eta * epsilon_z
tau_zx = c55_eta * gamma_zx

print("应力分量:")
print(f"σ_x = c11·ε_x + c13·ε_z = {sigma_x}")
print(f"σ_z = c13·ε_x + c33·ε_z = {sigma_z}")
print(f"τ_zx = c55·γ_zx = {tau_zx}")


4. 应力-应变关系（本构方程）
应力分量:
σ_x = c11·ε_x + c13·ε_z = c11(eta*h)*Derivative(u(x, z), x) + c13(eta*h)*Derivative(w(x, z), z)
σ_z = c13·ε_x + c33·ε_z = c13(eta*h)*Derivative(u(x, z), x) + c33(eta*h)*Derivative(w(x, z), z)
τ_zx = c55·γ_zx = (Derivative(u(x, z), z) + Derivative(w(x, z), x))*c55(eta*h)


In [33]:
print("\n5. 平衡方程")
print("=" * 60)

# 平衡方程1: ∂σ_x/∂x + ∂τ_zx/∂z = 0
balance_eq1 = sp.diff(sigma_x, x) + sp.diff(tau_zx, z)

# 平衡方程2: ∂τ_zx/∂x + ∂σ_z/∂z = 0  
balance_eq2 = sp.diff(tau_zx, x) + sp.diff(sigma_z, z)

print("平衡方程1:")
print(f"∂σ_x/∂x + ∂τ_zx/∂z = {balance_eq1} = 0")
print("\n平衡方程2:")
print(f"∂τ_zx/∂x + ∂σ_z/∂z = {balance_eq2} = 0")


5. 平衡方程
平衡方程1:
∂σ_x/∂x + ∂τ_zx/∂z = 0 = 0

平衡方程2:
∂τ_zx/∂x + ∂σ_z/∂z = 0 = 0


In [34]:
print("\n6. 代入位移函数并简化平衡方程")
print("=" * 60)

# 对平衡方程进行坐标变换和简化
balance_eq1_subs = balance_eq1.subs(coord_subs).simplify()
balance_eq2_subs = balance_eq2.subs(coord_subs).simplify()

print("坐标变换后的平衡方程1:")
print(f"Eq1 = {balance_eq1_subs}")
print("\n坐标变换后的平衡方程2:")
print(f"Eq2 = {balance_eq2_subs}")


6. 代入位移函数并简化平衡方程
坐标变换后的平衡方程1:
Eq1 = 0

坐标变换后的平衡方程2:
Eq2 = 0


In [35]:
print("\n7. 提取关于U(η)和W(η)的微分方程")
print("=" * 60)

# 定义导数符号
U_prime = sp.diff(U, eta)
U_dprime = sp.diff(U_prime, eta) 
W_prime = sp.diff(W, eta)
W_dprime = sp.diff(W_prime, eta)

# 材料常数导数
c11_prime = sp.diff(c11_eta, eta)
c13_prime = sp.diff(c13_eta, eta)
c33_prime = sp.diff(c33_eta, eta)
c55_prime = sp.diff(c55_eta, eta)

# 波数
k = n * sp.pi / L

print(f"波数 k = nπ/L = {k}")

# 平衡方程1的系数整理
print("\n平衡方程1的系数:")
eq1_coeff = balance_eq1_subs / (h * sp.cos(n*sp.pi*xi))
eq1_coeff = sp.collect(sp.expand(eq1_coeff), [U, U_prime, U_dprime, W, W_prime, W_dprime])
print(f"Eq1_coeff = {eq1_coeff}")

# 平衡方程2的系数整理
print("\n平衡方程2的系数:")
eq2_coeff = balance_eq2_subs / (h * sp.sin(n*sp.pi*xi))  
eq2_coeff = sp.collect(sp.expand(eq2_coeff), [U, U_prime, U_dprime, W, W_prime, W_dprime])
print(f"Eq2_coeff = {eq2_coeff}")


7. 提取关于U(η)和W(η)的微分方程
波数 k = nπ/L = pi*n/L

平衡方程1的系数:
Eq1_coeff = 0

平衡方程2的系数:
Eq2_coeff = 0
